# Legal NER (L-NER) with small from-scratch language models — parameter-efficient fine-tuning study

This notebook contains the code for **11 Phase-2 experiments** of my CS781 capstone (Indian legal NER, strict
span-level macro-F1): the parameter-efficient methods, the knowledge-distillation pair, the encoder-decoder
model and the backbone ablations. All backbones (~100M parameters) were **pre-trained from scratch** on a legal
corpus in Phase 1 with my own byte-level BPE tokenizer; this notebook only does the NER fine-tuning.

| run | backbone | method | trainable params | dev macro-F1 | **test macro-F1** |
|---|---|---|---|---|---|
| m1_normgain_cw | M1 dense decoder | RMSNorm gains only (BitFit analogue) + class weights | 0.04M | 0.405 | **0.514** |
| nokd_student_m1x6 | M1, 6 of 12 layers | full FT, no teacher (KD control) | 62.4M | 0.509 | **0.511** |
| kd_student_m1x6 | M1, 6 of 12 layers | knowledge distillation from M1+LoRA | 62.4M | 0.440 | **0.506** |
| m1_prompt_cw | M1 | prompt tuning (20 soft tokens) + class weights | 0.03M | 0.439 | **0.503** |
| m4_adapter | M4 (M1 + copy head) | bottleneck adapters | 1.21M | 0.467 | **0.495** |
| m2_lorma_plus | M2 (MLA + MoE) | LoRMA+ (multiplicative low-rank) | ~0.2M | 0.431 | **0.484** |
| m1_lorma_pi | M1 | LoRMAπ (rank-inflated multiplicative) | 0.22M | 0.401 | **0.483** |
| abl_noqk_lora_cw | ablation backbone: no QK-norm | LoRA + class weights | 0.26M | 0.441 | **0.472** |
| abl_untied_lora_cw | ablation backbone: untied embeddings | LoRA + class weights | 0.26M | 0.441 | **0.457** |
| abl_base_lora_cw | ablation backbone: base | LoRA + class weights | 0.26M | 0.407 | **0.437** |
| m3_full | M3 encoder (of encoder-decoder) | full fine-tuning | — | 0.268 | **0.256** |

*Dev = 10 held-out training documents (noisy); test = hidden competition test set (35 documents).*

**Structure**
1. Source modules (written to `src/` with `%%writefile`): model architectures, the NER pipeline, PEFT methods.
2. Setup: data, tokenisation, class weights, macro-F1 model selection.
3. Experiment driver.
4. The 11 runs.

**Requirements:** PyTorch ≥ 2.1 with a GPU, `tokenizers`, `numpy`. To actually *run* it you need the Phase-1
checkpoints (`best.pt` files), the tokenizer JSON and the L-NER data — set the paths in Section 2. Without them
the notebook is still a complete, readable reference implementation.

**Key techniques implemented here:** exact character-offset mapping through text normalisation, BIO tagging with
punctuation-split tokens, sliding windows, word-consistent constrained Viterbi decoding, class-weighted CE,
LoRA / LoRMA+ / LoRMAπ / adapters / prompt tuning / norm-gain tuning, bidirectional attention for decoder
backbones, and logit distillation into a layer-dropped student.

## 1 · Source modules

* `common.py` – text normalisation used at pre-training time (ASCII folding, punctuation map).
* `lm_common.py` – shared transformer pieces: RMSNorm, RoPE, GQA attention with QK-norm, SwiGLU, KV cache.
* `model_dense.py` (M1), `model_mla_moe.py` (M2: Multi-head Latent Attention + DeepSeek-style MoE),
  `model_encdec.py` (M3), `model_copy.py` (M4: M1 + learned copy/pointer head).
* `model_registry.py` – `build_model(arch, cfg)` (replaces the pre-training script's registry).
* `p2_ner.py` – offsets ↔ tokens, BIO tags, windows, (word-level) Viterbi decoding, strict F1.
* `p2_model.py` – token tagger on any backbone, windowed training / prediction.
* `p2_peft.py` – LoRA, LoRMA+, LoRMAπ, adapters, prompt tuning, norm-gain tuning, layer dropping, KD.

In [ ]:
%%writefile src/common.py
"""
common.py -- rules shared by every Phase-1 stage (tokenizer, sharding, validation, generation).

normalise(text)
    Make training text look like the official test prompts: ASCII only, single spaces,
    no newlines (Cell 5: 0/1000 test prompts contain a newline, double space or non-ASCII).
    Common typographic characters are mapped to ASCII first (curly quotes -> straight, dashes
    -> '-', ellipsis -> '...'); remaining accents are stripped via NFKD; anything else dropped.

split_of(norm_text, source)
    Deterministic DOCUMENT-level train/val assignment from a hash of the full normalised text.
    Hashing the text (not a record index) sends exact duplicates to the SAME split, so a
    duplicated judgment can never sit in train and val at once.
"""
from __future__ import annotations

import hashlib
import unicodedata

PUNCT_MAP = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "′": "'",
    "“": '"', "”": '"', "„": '"', "″": '"',
    "–": "-", "—": "-", "‒": "-", "−": "-", "‐": "-", "‑": "-",
    "…": "...", " ": " ", "­": "", "•": "*", "·": ".", "​": "",
    "﻿": "",
})

SOURCES = ["acts", "supreme_court", "tribunals", "high_courts"]

# Fraction of documents held out for validation, per source. SC is the test domain, so it
# gets the largest share (~1,100 docs); the big sources need only a small fraction.
VAL_FRAC = {"supreme_court": 0.02, "acts": 0.02, "tribunals": 0.002, "high_courts": 0.002}


def normalise(text: str) -> str:
    if not text:
        return ""
    if not text.isascii():
        text = unicodedata.normalize("NFKD", text.translate(PUNCT_MAP))
        text = text.encode("ascii", "ignore").decode("ascii")
    return " ".join(text.split())


def doc_hash(norm_text: str) -> int:
    return int.from_bytes(hashlib.blake2b(norm_text.encode(), digest_size=8).digest(), "big")


def split_of(norm_text: str, source: str) -> str:
    """'val' or 'train', stable across runs/machines (no Python hash randomisation)."""
    return "val" if (doc_hash(norm_text) % 1_000_000) < VAL_FRAC[source] * 1_000_000 else "train"

In [ ]:
%%writefile src/lm_common.py
"""
lm_common.py -- building blocks shared by all Phase-1 architectures + KV-cached generation.

Design choices (T4-aware):
  * RMSNorm (cheaper than LayerNorm, no mean subtraction), computed in fp32 for fp16 stability.
  * RoPE positions (no learned position table; extrapolates a little beyond training length).
  * QK-norm: RMSNorm on per-head queries/keys before RoPE. Bounds attention logits, which is
    the classic fp16 overflow point. The T4 has no bf16, so we train in fp16.
  * SwiGLU feed-forward (gated MLP; better quality per parameter than GELU MLP).
  * Attention via F.scaled_dot_product_attention. On a T4 (sm_75) PyTorch dispatches to the
    memory-efficient kernel (FlashAttention-2 needs sm_80+): O(T) memory, never builds the
    T x T matrix.
  * KV cache: prompt is processed once (prefill), then one token per step reuses cached K/V.
"""
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                              # fp32 statistics
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return xf.to(x.dtype) * self.weight.to(x.dtype)


def rope_cache(seq_len: int, head_dim: int, theta: float = 10000.0, device=None):
    """cos/sin tables of shape (seq_len, head_dim/2), fp32."""
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    ang = torch.outer(torch.arange(seq_len, device=device).float(), inv)
    return ang.cos(), ang.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotate-half formulation, computed in fp32."""
    h = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :h], xf[..., h:]
    c, s = cos[None, None], sin[None, None]
    return torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)


@dataclass
class KVCache:
    k: torch.Tensor          # (B, n_kv_heads, max_T, head_dim)
    v: torch.Tensor


class Attention(nn.Module):
    """Multi-head attention with grouped-query K/V (n_kv_heads <= n_heads), QK-norm, RoPE."""

    def __init__(self, d_model, n_heads, n_kv_heads, qk_norm=True):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.h, self.kv, self.hd = n_heads, n_kv_heads, d_model // n_heads
        self.wq = nn.Linear(d_model, n_heads * self.hd, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.hd, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.hd, bias=False)
        self.wo = nn.Linear(n_heads * self.hd, d_model, bias=False)
        self.qk_norm = qk_norm
        if qk_norm:
            self.q_norm, self.k_norm = RMSNorm(self.hd), RMSNorm(self.hd)

    def forward(self, x, cos, sin, cache: KVCache | None = None, pos: int = 0):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.wk(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.wv(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        if self.qk_norm:
            q, k = self.q_norm(q), self.k_norm(k)
        q = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T])
        k = apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
        if cache is not None:                                        # write new K/V, read all
            cache.k[:, :, pos:pos + T] = k
            cache.v[:, :, pos:pos + T] = v
            k, v = cache.k[:, :, :pos + T], cache.v[:, :, :pos + T]
        if self.kv != self.h:                                        # GQA: share K/V across groups
            rep = self.h // self.kv
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        # causal mask only needed when several queries are processed at once; this code path
        # is used for training (pos=0) and prefill (pos=0). Decoding uses T=1 (sees all past).
        assert T == 1 or pos == 0, "chunked prefill (T>1 at pos>0) is not supported"
        y = F.scaled_dot_product_attention(q, k, v, is_causal=(T > 1))
        return self.wo(y.transpose(1, 2).reshape(B, T, self.h * self.hd))


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden, bias=False)   # gate
        self.w3 = nn.Linear(d_model, hidden, bias=False)   # up
        self.w2 = nn.Linear(hidden, d_model, bias=False)   # down

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


def count_params(model: nn.Module) -> int:
    """Unique parameters (tied embedding counted once)."""
    return sum(p.numel() for p in {id(p): p for p in model.parameters()}.values())


# --------------------------------------------------------------------------- generation
def _apply_sampling(logits, temperature, top_k, top_p):
    if temperature <= 0:
        return logits.argmax(-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1).values[..., -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p and top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = sl.softmax(-1).cumsum(-1)
        remove = cp - sl.softmax(-1) > top_p                          # keep first token always
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(-1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1)


@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.0, top_k=0, top_p=1.0,
             repetition_penalty=1.0, banned_ids=(0,), no_repeat_ngram=0,
             first_allowed=None, ngram_window=384):
    """KV-cached autoregressive generation for a batch of EQUAL-length prompts.
    idx: (B, T) prompt ids. Returns (B, max_new_tokens) new ids.
    banned_ids: <|endoftext|> by default, so the model cannot stop before 128 words.
    repetition_penalty: applied only to tokens the model has GENERATED so far -- penalising
      prompt tokens would push the model away from party names / sections it should reuse.
    first_allowed: optional bool (V,) mask of tokens allowed as the FIRST generated token
      (we pass "starts with a space" so the continuation begins a new word, like the reference).
    ngram_window: no-repeat-ngram looks back this many tokens (bounds the Python cost)."""
    model.eval()
    B, T = idx.shape
    max_T = T + max_new_tokens
    # cache dtype must match the K/V activations: fp16 under autocast, else the weights' dtype
    cdt = torch.float16 if torch.is_autocast_enabled() else next(model.parameters()).dtype
    caches = model.make_caches(B, max_T, idx.device, dtype=cdt)
    logits = model(idx, caches=caches, pos=0)[:, -1].float()         # prefill
    out = []
    seen = idx.clone()
    for i in range(max_new_tokens):
        if repetition_penalty != 1.0 and i > 0:                       # CTRL-style, generated tokens only
            gen = seen[:, T:]
            g = logits.gather(1, gen)
            logits.scatter_(1, gen, torch.where(g > 0, g / repetition_penalty, g * repetition_penalty))
        if banned_ids:
            logits[:, list(banned_ids)] = float("-inf")
        if i == 0 and first_allowed is not None:
            logits[:, ~first_allowed] = float("-inf")
        if no_repeat_ngram and seen.size(1) >= no_repeat_ngram:
            n = no_repeat_ngram
            for b in range(B):                                        # block repeated n-grams
                s = seen[b, -ngram_window:].tolist(); prefix = tuple(s[-(n - 1):])
                bad = [s[j + n - 1] for j in range(len(s) - n + 1) if tuple(s[j:j + n - 1]) == prefix]
                if bad:
                    logits[b, bad] = float("-inf")
        nxt = _apply_sampling(logits, temperature, top_k, top_p)      # (B, 1)
        out.append(nxt)
        seen = torch.cat([seen, nxt], 1)
        if i + 1 < max_new_tokens:
            logits = model(nxt, caches=caches, pos=T + i)[:, -1].float()
    return torch.cat(out, 1)

In [ ]:
%%writefile src/model_dense.py
"""
model_dense.py -- M1: dense decoder-only Transformer (the controlled baseline).

Pre-norm blocks:  x = x + Attn(RMSNorm(x));  x = x + SwiGLU(RMSNorm(x))
GQA attention (n_kv_heads < n_heads) shrinks the KV cache for generation; QK-norm + RoPE;
tied input/output embeddings (A1 Q3 studied tying input/output matrices in word2vec).
Every other architecture (M2-M4) is compared against this one at the same token budget.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F

from lm_common import RMSNorm, Attention, SwiGLU, KVCache, rope_cache


@dataclass
class DenseConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 4
    ffn_hidden: int = 2048          # ~8/3 * d_model, multiple of 64 (tensor cores)
    max_seq_len: int = 2048         # RoPE table length (training uses 1024)
    rope_theta: float = 10000.0
    qk_norm: bool = True
    tie_embeddings: bool = True
    init_std: float = 0.02


class Block(nn.Module):
    def __init__(self, c: DenseConfig):
        super().__init__()
        self.norm1 = RMSNorm(c.d_model)
        self.attn = Attention(c.d_model, c.n_heads, c.n_kv_heads, c.qk_norm)
        self.norm2 = RMSNorm(c.d_model)
        self.mlp = SwiGLU(c.d_model, c.ffn_hidden)

    def forward(self, x, cos, sin, cache=None, pos=0):
        x = x + self.attn(self.norm1(x), cos, sin, cache, pos)
        return x + self.mlp(self.norm2(x))


class DenseGPT(nn.Module):
    def __init__(self, c: DenseConfig):
        super().__init__()
        self.cfg = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layers)])
        self.norm_f = RMSNorm(c.d_model)
        self.lm_head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        if c.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = rope_cache(c.max_seq_len, c.d_model // c.n_heads, c.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        # GPT-2 trick: scale residual-output projections by 1/sqrt(2 * n_layers)
        for n, p in self.named_parameters():
            if n.endswith("wo.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, c.init_std / math.sqrt(2 * c.n_layers))

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, self.cfg.init_std)

    def make_caches(self, B, max_T, device, dtype=torch.float16):
        hd, kv = self.cfg.d_model // self.cfg.n_heads, self.cfg.n_kv_heads
        return [KVCache(torch.zeros(B, kv, max_T, hd, device=device, dtype=dtype),
                        torch.zeros(B, kv, max_T, hd, device=device, dtype=dtype))
                for _ in self.blocks]

    def forward(self, idx, targets=None, caches=None, pos: int = 0):
        x = self.tok_emb(idx)
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope_cos, self.rope_sin, None if caches is None else caches[i], pos)
        x = self.norm_f(x)
        if targets is None:
            return self.lm_head(x)                                    # (B, T, V) logits
        logits = self.lm_head(x)
        return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1),
                               ignore_index=-1)

In [ ]:
%%writefile src/model_mla_moe.py
"""
model_mla_moe.py -- M2: DeepSeek-V2/V3-style decoder: Multi-head Latent Attention (MLA) +
DeepSeekMoE (fine-grained routed experts + shared expert) with auxiliary-loss-free balancing.

Why this architecture under the capstone constraints
  * The 50-300M rule caps TOTAL parameters, but training cost scales with ACTIVE parameters.
    MoE spends the cap (~270M total) while each token only uses ~100M -- the same compute as M1,
    so M1 vs M2 is a compute-matched comparison of "dense vs sparse".
  * MLA caches ONE low-rank latent c_kv (d_c=256) + a small shared RoPE key (32) per token instead
    of per-head K and V: 288 numbers/token/layer vs 512 for M1's GQA and 1,536 for full MHA.
    During decoding the up-projections are ABSORBED into the query / output (no K/V is ever
    materialised), which is what makes large-N sampling (MBR) cheap.

MLA (per layer)                                  DeepSeekMoE FFN (layers 1..L-1; layer 0 dense)
  q      = W_q x            -> [q_nope | q_rope]    s_i   = sigmoid(u . e_i)          affinities
  c_kv   = RMSNorm(W_dkv x)       (d_c latent)      top-k by (s_i + b_i)              b_i: bias
  k_rope = RoPE(W_kr x)     (shared by all heads)   g_i   = s_i / sum_topk s_j        gate
  k_nope = W_uk c_kv ; v = W_uv c_kv                y     = Shared(x) + sum g_i E_i(x)
  decode: score = (W_uk^T q_nope) . c_kv + q_rope . k_rope ; out = W_uv (softmax . c_kv)
Balancing (DeepSeek-V3, aux-loss-free): after every optimizer step, b_i += gamma*sign(mean_load -
load_i) -- steers routing without adding a loss term that fights the LM objective.
Expert dispatch uses fixed per-expert capacity with sort-based scatter/gather + batched matmuls,
so every tensor has a static shape (torch.compile-friendly, one bmm instead of 16 small GEMMs).
"""
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F

from lm_common import RMSNorm, SwiGLU, rope_cache, apply_rope


@dataclass
class MLAMoEConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    qk_nope_dim: int = 32          # per-head key/query part WITHOUT position (from the latent)
    qk_rope_dim: int = 32          # per-head decoupled RoPE part (key shared across heads)
    v_head_dim: int = 64           # nope + rope == v dim -> SDPA's efficient kernel applies
    kv_latent_dim: int = 256       # d_c: the only K/V state that is cached
    n_routed_experts: int = 16
    n_shared_experts: int = 1
    top_k: int = 3                 # shared(1) + routed(3) = 4 x 512 = 2048 active hidden (= M1)
    expert_hidden: int = 512       # fine-grained experts (M1's 2048-wide FFN split 4 ways)
    first_dense_layers: int = 1    # layer 0 keeps a dense FFN (stabilises early routing)
    dense_ffn_hidden: int = 2048
    capacity_factor: float = 1.25  # training-time slots per expert = 1.25 x average load
    bias_update_rate: float = 1e-3 # gamma of the aux-loss-free balancing
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    tie_embeddings: bool = True
    init_std: float = 0.02


@dataclass
class MLACache:
    c: torch.Tensor                # (B, max_T, d_c)        compressed KV latent
    kr: torch.Tensor               # (B, 1, max_T, rope)    shared RoPE key


class MLAttention(nn.Module):
    def __init__(self, c: MLAMoEConfig):
        super().__init__()
        self.h, self.nope, self.rope, self.vd, self.dc = (c.n_heads, c.qk_nope_dim, c.qk_rope_dim,
                                                          c.v_head_dim, c.kv_latent_dim)
        qk = self.nope + self.rope
        assert qk == self.vd, "keep qk head dim == v head dim so SDPA can use its fused kernel"
        self.wq = nn.Linear(c.d_model, self.h * qk, bias=False)
        self.w_dkv = nn.Linear(c.d_model, self.dc, bias=False)       # down-projection to latent
        self.w_kr = nn.Linear(c.d_model, self.rope, bias=False)      # decoupled RoPE key
        self.w_uk = nn.Linear(self.dc, self.h * self.nope, bias=False)  # latent -> keys
        self.w_uv = nn.Linear(self.dc, self.h * self.vd, bias=False)    # latent -> values
        self.wo = nn.Linear(self.h * self.vd, c.d_model, bias=False)
        self.q_norm, self.kv_norm, self.kr_norm = RMSNorm(qk), RMSNorm(self.dc), RMSNorm(self.rope)
        self.scale = qk ** -0.5

    def forward(self, x, cos, sin, cache: MLACache | None = None, pos: int = 0):
        B, T, _ = x.shape
        q = self.q_norm(self.wq(x).view(B, T, self.h, self.nope + self.rope).transpose(1, 2))
        q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
        q_rope = apply_rope(q_rope, cos[pos:pos + T], sin[pos:pos + T])
        c_kv = self.kv_norm(self.w_dkv(x))                                      # (B, T, d_c)
        k_rope = apply_rope(self.kr_norm(self.w_kr(x)).unsqueeze(1),
                            cos[pos:pos + T], sin[pos:pos + T])                 # (B, 1, T, rope)
        if cache is not None:
            cache.c[:, pos:pos + T] = c_kv
            cache.kr[:, :, pos:pos + T] = k_rope

        if T > 1 or cache is None:       # training / prefill: materialise K,V once, fused SDPA
            assert pos == 0, "chunked prefill is not supported"
            k_nope = self.w_uk(c_kv).view(B, T, self.h, self.nope).transpose(1, 2)
            v = self.w_uv(c_kv).view(B, T, self.h, self.vd).transpose(1, 2)
            k = torch.cat([k_nope, k_rope.expand(B, self.h, T, self.rope)], dim=-1)
            y = F.scaled_dot_product_attention(torch.cat([q_nope, q_rope], -1), k, v, is_causal=T > 1)
        else:                            # decoding: ABSORBED form, attends straight to the latents
            C = cache.c[:, :pos + 1]                                            # (B, S, d_c)
            Kr = cache.kr[:, 0, :pos + 1]                                       # (B, S, rope)
            w_uk = self.w_uk.weight.view(self.h, self.nope, self.dc)
            w_uv = self.w_uv.weight.view(self.h, self.vd, self.dc)
            q_lat = torch.einsum("bhtn,hnc->bhtc", q_nope, w_uk.to(q_nope.dtype))   # W_uk^T q
            s = (torch.einsum("bhtc,bsc->bhts", q_lat, C.to(q_lat.dtype)) +
                 torch.einsum("bhtr,bsr->bhts", q_rope, Kr.to(q_rope.dtype)))
            p = torch.softmax(s.float() * self.scale, dim=-1).to(q_lat.dtype)
            ctx = torch.einsum("bhts,bsc->bhtc", p, C.to(p.dtype))             # (B, H, 1, d_c)
            y = torch.einsum("bhtc,hvc->bhtv", ctx, w_uv.to(ctx.dtype))         # W_uv . ctx
        return self.wo(y.transpose(1, 2).reshape(B, T, self.h * self.vd))


class DeepSeekMoE(nn.Module):
    def __init__(self, c: MLAMoEConfig):
        super().__init__()
        E, d, h = c.n_routed_experts, c.d_model, c.expert_hidden
        self.E, self.k, self.cap = E, c.top_k, c.capacity_factor
        self.shared = SwiGLU(d, h * c.n_shared_experts)
        self.router = nn.Linear(d, E, bias=False)
        self.w1 = nn.Parameter(torch.empty(E, d, h))     # gate
        self.w3 = nn.Parameter(torch.empty(E, d, h))     # up
        self.w2 = nn.Parameter(torch.empty(E, h, d))     # down
        self.register_buffer("router_bias", torch.zeros(E))                   # saved, not trained
        self.register_buffer("load_accum", torch.zeros(E), persistent=False)  # tokens per expert
        self.register_buffer("drop_accum", torch.zeros(2), persistent=False)  # (dropped, total)

    def forward(self, x):
        B, T, d = x.shape
        N, E, k = B * T, self.E, self.k
        xf = x.reshape(N, d)
        with torch.autocast("cuda", enabled=False):                          # routing in fp32
            aff = torch.sigmoid(xf.float() @ self.router.weight.float().t())  # (N, E)
        _, idx = torch.topk(aff + self.router_bias, k, dim=-1)                # bias only SELECTS
        gate = aff.gather(1, idx)
        gate = gate / gate.sum(-1, keepdim=True)                              # (N, k)

        flat_e = idx.reshape(-1)                                              # (N*k,)
        flat_tok = torch.arange(N, device=x.device).repeat_interleave(k)
        C = N if not self.training else int(math.ceil(self.cap * N * k / E))  # no drops at eval
        counts = torch.zeros(E, device=x.device).scatter_add_(0, flat_e, torch.ones_like(flat_e, dtype=torch.float))
        order = torch.argsort(flat_e, stable=True)                            # group by expert
        se, tok = flat_e[order], flat_tok[order]
        starts = torch.cumsum(counts, 0) - counts
        slot = torch.arange(N * k, device=x.device) - starts[se].long()       # position in expert
        keep = slot < C
        dest = torch.where(keep, se * C + slot, torch.full_like(se, E * C))   # overflow -> dummy row

        buf = xf.new_zeros(E * C + 1, d)
        buf[dest] = xf[tok]                                                   # dispatch
        h = buf[:E * C].view(E, C, d)
        h = F.silu(torch.bmm(h, self.w1)) * torch.bmm(h, self.w3)              # all experts, 1 bmm
        o = torch.bmm(h, self.w2).reshape(E * C, d)
        o = torch.cat([o, o.new_zeros(1, d)], 0)
        wts = (gate.reshape(-1)[order] * keep).to(o.dtype).unsqueeze(1)
        y = torch.zeros(N, d, device=x.device, dtype=o.dtype).index_add_(0, tok, o[dest] * wts)  # combine

        if self.training:
            self.load_accum += counts
            self.drop_accum += torch.stack([(~keep).sum().float(), xf.new_tensor(float(N * k))])
        return self.shared(x) + y.view(B, T, d).to(x.dtype)


class Block(nn.Module):
    def __init__(self, c: MLAMoEConfig, dense: bool):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(c.d_model), RMSNorm(c.d_model)
        self.attn = MLAttention(c)
        self.ffn = SwiGLU(c.d_model, c.dense_ffn_hidden) if dense else DeepSeekMoE(c)

    def forward(self, x, cos, sin, cache=None, pos=0):
        x = x + self.attn(self.norm1(x), cos, sin, cache, pos)
        return x + self.ffn(self.norm2(x))


class MLAMoEGPT(nn.Module):
    def __init__(self, c: MLAMoEConfig):
        super().__init__()
        self.cfg = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c, dense=i < c.first_dense_layers) for i in range(c.n_layers)])
        self.norm_f = RMSNorm(c.d_model)
        self.lm_head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        if c.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = rope_cache(c.max_seq_len, c.qk_rope_dim, c.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.moe_layers = [b.ffn for b in self.blocks if isinstance(b.ffn, DeepSeekMoE)]
        self.apply(self._init)
        for n, p in self.named_parameters():               # expert weights + scaled residual outputs
            if n.endswith((".w1", ".w3")):
                nn.init.normal_(p, 0.0, c.init_std)
            if n.endswith((".w2", "wo.weight", "w2.weight")):
                nn.init.normal_(p, 0.0, c.init_std / math.sqrt(2 * c.n_layers))

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, self.cfg.init_std)

    def active_params(self):
        """Parameters used per token (shared + top-k routed experts, not all 16)."""
        c = self.cfg
        total = sum(p.numel() for p in {id(p): p for p in self.parameters()}.values())
        per_expert = 3 * c.d_model * c.expert_hidden
        inactive = len(self.moe_layers) * (c.n_routed_experts - c.top_k) * per_expert
        return total - inactive

    def make_caches(self, B, max_T, device, dtype=torch.float16):
        c = self.cfg
        return [MLACache(torch.zeros(B, max_T, c.kv_latent_dim, device=device, dtype=dtype),
                         torch.zeros(B, 1, max_T, c.qk_rope_dim, device=device, dtype=dtype))
                for _ in self.blocks]

    @torch.no_grad()
    def after_step(self):
        """Aux-loss-free balancing (DeepSeek-V3): nudge each expert's routing bias towards equal
        load. Loads are summed over DDP ranks so every rank applies the identical update."""
        if not self.moe_layers:
            return {}
        loads = torch.stack([m.load_accum for m in self.moe_layers])          # (L, E)
        drops = torch.stack([m.drop_accum for m in self.moe_layers]).sum(0)   # (dropped, total)
        if dist.is_available() and dist.is_initialized():
            dist.all_reduce(loads); dist.all_reduce(drops)
        mean = loads.mean(1, keepdim=True)
        for m, l, mu in zip(self.moe_layers, loads, mean):
            m.router_bias += self.cfg.bias_update_rate * torch.sign(mu - l)
            m.load_accum.zero_(); m.drop_accum.zero_()
        max_over_mean = (loads.max(1).values / mean.squeeze(1).clamp_min(1)).mean()
        return {"moe_max/mean": round(max_over_mean.item(), 3),
                "drop%": round(100 * (drops[0] / drops[1].clamp_min(1)).item(), 2)}

    def forward(self, idx, targets=None, caches=None, pos: int = 0):
        x = self.tok_emb(idx)
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope_cos, self.rope_sin, None if caches is None else caches[i], pos)
        logits = self.lm_head(self.norm_f(x))
        if targets is None:
            return logits
        return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-1)

In [ ]:
%%writefile src/model_encdec.py
# model_encdec.py -- M3: encoder-decoder Transformer trained on "prompt -> continuation".
# Each 1,024-token training window is split: the first enc_len=640 tokens go to a BIDIRECTIONAL
# encoder, the decoder (causal self-attn + cross-attn to the encoder) predicts the next 384.
# This mirrors the task exactly (read a 512-word prompt, write what comes next) and gives an
# encoder whose every token sees the whole prompt -- useful again in Phases 2-3 (NER, roles).
# Generation: the prompt is encoded ONCE, each decoder layer's cross-attention K/V is computed
# once and cached, and decoder self-attention uses a normal KV cache.
from __future__ import annotations
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from lm_common import RMSNorm, SwiGLU, rope_cache, apply_rope


@dataclass
class EncDecConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_enc_layers: int = 6
    n_dec_layers: int = 6
    n_heads: int = 12
    n_kv_heads: int = 4
    ffn_hidden: int = 2048
    enc_len: int = 640             # encoder part of each training window (prompt side)
    bos_id: int = 5                # <|bos|> starts every decoder sequence
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    init_std: float = 0.02


@dataclass
class DecCache:
    k: torch.Tensor                # decoder self-attention keys   (B, kv, max_T, hd)
    v: torch.Tensor
    ck: torch.Tensor = None        # cross-attention keys/values from the encoder (B, kv, Te, hd)
    cv: torch.Tensor = None


class SelfAttn(nn.Module):
    # GQA self-attention with QK-norm + RoPE; causal (decoder) or bidirectional (encoder)
    def __init__(self, c: EncDecConfig, causal: bool):
        super().__init__()
        self.h, self.kv, self.hd, self.causal = c.n_heads, c.n_kv_heads, c.d_model // c.n_heads, causal
        self.wq = nn.Linear(c.d_model, self.h * self.hd, bias=False)
        self.wk = nn.Linear(c.d_model, self.kv * self.hd, bias=False)
        self.wv = nn.Linear(c.d_model, self.kv * self.hd, bias=False)
        self.wo = nn.Linear(self.h * self.hd, c.d_model, bias=False)
        self.q_norm, self.k_norm = RMSNorm(self.hd), RMSNorm(self.hd)

    def forward(self, x, cos, sin, cache=None, pos=0):
        B, T, _ = x.shape
        q = self.q_norm(self.wq(x).view(B, T, self.h, self.hd).transpose(1, 2))
        k = self.k_norm(self.wk(x).view(B, T, self.kv, self.hd).transpose(1, 2))
        v = self.wv(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T]), apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
        if cache is not None:                                        # decoder generation
            cache.k[:, :, pos:pos + T], cache.v[:, :, pos:pos + T] = k, v
            k, v = cache.k[:, :, :pos + T], cache.v[:, :, :pos + T]
        rep = self.h // self.kv
        k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        assert T == 1 or pos == 0
        y = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and T > 1)
        return self.wo(y.transpose(1, 2).reshape(B, T, self.h * self.hd))


class CrossAttn(nn.Module):
    # decoder queries attend to encoder states (no mask, no RoPE); K/V computable once per prompt
    def __init__(self, c: EncDecConfig):
        super().__init__()
        self.h, self.kv, self.hd = c.n_heads, c.n_kv_heads, c.d_model // c.n_heads
        self.wq = nn.Linear(c.d_model, self.h * self.hd, bias=False)
        self.wk = nn.Linear(c.d_model, self.kv * self.hd, bias=False)
        self.wv = nn.Linear(c.d_model, self.kv * self.hd, bias=False)
        self.wo = nn.Linear(self.h * self.hd, c.d_model, bias=False)
        self.q_norm, self.k_norm = RMSNorm(self.hd), RMSNorm(self.hd)

    def mem_kv(self, mem):
        B, S, _ = mem.shape
        k = self.k_norm(self.wk(mem).view(B, S, self.kv, self.hd).transpose(1, 2))
        v = self.wv(mem).view(B, S, self.kv, self.hd).transpose(1, 2)
        return k, v

    def forward(self, x, k, v):
        B, T, _ = x.shape
        q = self.q_norm(self.wq(x).view(B, T, self.h, self.hd).transpose(1, 2))
        rep = self.h // self.kv
        y = F.scaled_dot_product_attention(q, k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1))
        return self.wo(y.transpose(1, 2).reshape(B, T, self.h * self.hd))


class EncLayer(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.n2 = RMSNorm(c.d_model), RMSNorm(c.d_model)
        self.attn, self.mlp = SelfAttn(c, causal=False), SwiGLU(c.d_model, c.ffn_hidden)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)
        return x + self.mlp(self.n2(x))


class DecLayer(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.n2, self.n3 = RMSNorm(c.d_model), RMSNorm(c.d_model), RMSNorm(c.d_model)
        self.self_attn, self.cross = SelfAttn(c, causal=True), CrossAttn(c)
        self.mlp = SwiGLU(c.d_model, c.ffn_hidden)

    def forward(self, x, cos, sin, ck, cv, cache=None, pos=0):
        x = x + self.self_attn(self.n1(x), cos, sin, cache, pos)
        x = x + self.cross(self.n2(x), ck, cv)
        return x + self.mlp(self.n3(x))


class EncDecLM(nn.Module):
    def __init__(self, c: EncDecConfig):
        super().__init__()
        self.cfg = c
        self.max_prompt_tokens = c.enc_len                     # inference keeps the last 640 tokens
        self.tok_emb = nn.Embedding(c.vocab_size, c.d_model)  # shared by encoder, decoder, output
        self.enc = nn.ModuleList([EncLayer(c) for _ in range(c.n_enc_layers)])
        self.dec = nn.ModuleList([DecLayer(c) for _ in range(c.n_dec_layers)])
        self.enc_norm, self.dec_norm = RMSNorm(c.d_model), RMSNorm(c.d_model)
        self.lm_head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        cos, sin = rope_cache(c.max_seq_len, c.d_model // c.n_heads, c.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        L = c.n_enc_layers + c.n_dec_layers
        for n, p in self.named_parameters():                   # scaled residual-output init
            if n.endswith(("wo.weight", "w2.weight")):
                nn.init.normal_(p, 0.0, c.init_std / math.sqrt(2 * L))
        self._prompt_len = 0

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, self.cfg.init_std)

    def encode(self, src):
        x = self.tok_emb(src)
        for layer in self.enc:
            x = layer(x, self.rope_cos, self.rope_sin)
        return self.enc_norm(x)

    def decode_tf(self, mem, dec_in):
        # teacher-forced decoder pass over the whole target sequence -> logits (B, Td, V)
        x = self.tok_emb(dec_in)
        for layer in self.dec:
            ck, cv = layer.cross.mem_kv(mem)
            x = layer(x, self.rope_cos, self.rope_sin, ck, cv)
        return self.lm_head(self.dec_norm(x))

    def make_caches(self, B, max_T, device, dtype=torch.float16):
        kv, hd = self.cfg.n_kv_heads, self.cfg.d_model // self.cfg.n_heads
        return [DecCache(torch.zeros(B, kv, max_T, hd, device=device, dtype=dtype),
                         torch.zeros(B, kv, max_T, hd, device=device, dtype=dtype)) for _ in self.dec]

    def _gen_step(self, idx, caches, pos):
        # pos == 0: idx is the PROMPT -> encode once, cache cross K/V, decode <|bos|>
        # pos  > 0: idx is the last generated token (generation.py passes pos = prompt_len + i)
        if pos == 0:
            mem = self.encode(idx)
            for layer, c in zip(self.dec, caches):
                c.ck, c.cv = layer.cross.mem_kv(mem)
            self._prompt_len = idx.size(1)
            tok, dpos = torch.full((idx.size(0), 1), self.cfg.bos_id, device=idx.device), 0
        else:
            tok, dpos = idx, pos - self._prompt_len + 1
        x = self.tok_emb(tok)
        for layer, c in zip(self.dec, caches):
            x = layer(x, self.rope_cos, self.rope_sin, c.ck, c.cv, c, dpos)
        return self.lm_head(self.dec_norm(x))

    def forward(self, idx, targets=None, caches=None, pos: int = 0):
        if caches is not None:
            return self._gen_step(idx, caches, pos)
        # training / eval: rebuild the (T+1)-token window, split it into prompt | continuation
        full = torch.cat([idx[:, :1], targets], dim=1)
        T, s = idx.size(1), self.cfg.enc_len
        src, tgt = full[:, :s], full[:, s:T]                                   # 640 | 384 tokens
        bos = torch.full((idx.size(0), 1), self.cfg.bos_id, device=idx.device, dtype=idx.dtype)
        logits = self.decode_tf(self.encode(src), torch.cat([bos, tgt[:, :-1]], 1))
        return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-1)

In [ ]:
%%writefile src/model_copy.py
# model_copy.py -- M4: dense decoder (M1 backbone) + a learned LEGAL COPY HEAD.
# Legal continuations reuse the prompt heavily (party names, section numbers, quoted phrases);
# the decoding study showed a fixed copy bonus helps up to beta~1 and then hurts. M4 LEARNS it.
# Output = mixture of two distributions, mixed by a per-token gate g_t = sigmoid(w . h_t + b):
#   p(y_t) = (1 - g_t) * softmax(W h_t)[y_t]            (generate from the vocabulary)
#          +      g_t  * sum_{i<t, y_i = y_t} a_{t,i}     (copy: point at an earlier position)
# a_{t,i} = softmax_i(q_t . k_i / sqrt(d_c)) over i < t, q_t = Wq h_t, k_i = Wk h_i.
# Induction-style pointer: key i is the state that PRECEDED token y_i = x_{i+1}, so the head
# learns "the context now looks like the context before that earlier token -> copy it".
# Strictly i < t, so the target y_t is never visible. Extra parameters: 2*768*128 + 769 ~ 0.2M.
from __future__ import annotations
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from model_dense import DenseGPT, DenseConfig

NEG = -1e4                                    # finite "minus infinity" (keeps gradients NaN-free)


@dataclass
class CopyConfig(DenseConfig):
    copy_dim: int = 128                       # query/key size of the pointer attention
    gate_bias_init: float = -2.0              # sigmoid(-2) ~ 0.12: start mostly generating


@dataclass
class CopyCache:
    k: torch.Tensor                           # pointer keys  (B, max_T, copy_dim)
    ids: torch.Tensor                         # input token ids (B, max_T); value of key i = ids[i+1]


class CopyGPT(DenseGPT):
    def __init__(self, c: CopyConfig):
        super().__init__(c)                                          # M1 backbone + its init
        self.copy_q = nn.Linear(c.d_model, c.copy_dim, bias=False)
        self.copy_k = nn.Linear(c.d_model, c.copy_dim, bias=False)
        self.gate = nn.Linear(c.d_model, 1)
        for m in (self.copy_q, self.copy_k, self.gate):
            nn.init.normal_(m.weight, 0.0, c.init_std)
        nn.init.constant_(self.gate.bias, c.gate_bias_init)

    def _hidden(self, idx, caches=None, pos=0):
        x = self.tok_emb(idx)
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope_cos, self.rope_sin, None if caches is None else caches[i], pos)
        return self.norm_f(x)

    def make_caches(self, B, max_T, device, dtype=torch.float16):
        # M1's per-layer KV caches + one pointer cache (keys + token ids) at the end
        return super().make_caches(B, max_T, device, dtype) + [
            CopyCache(torch.zeros(B, max_T, self.cfg.copy_dim, device=device, dtype=dtype),
                      torch.zeros(B, max_T, dtype=torch.long, device=device))]

    def copy_loss(self, h, targets):
        """Mean NLL of the gated mixture (a proper probability -> comparable to M1's loss)."""
        B, T, _ = h.shape
        z = self.lm_head(h)
        lp_vocab = -F.cross_entropy(z.float().view(-1, z.size(-1)), targets.reshape(-1),
                                    reduction="none").view(B, T)           # log p_vocab(y_t)
        with torch.autocast("cuda", enabled=False):                        # pointer math in fp32
            hf = h.float()
            allowed = torch.ones(T, T, dtype=torch.bool, device=h.device).tril(-1)  # i < t
            g = self.gate(hf).squeeze(-1).masked_fill(~allowed.any(-1), NEG)  # t=0: nothing to copy
            s = self.copy_q(hf) @ self.copy_k(hf).transpose(1, 2) / math.sqrt(self.cfg.copy_dim)
            la = torch.log_softmax(s.masked_fill(~allowed, NEG), -1)       # log a_{t,i}
            match = targets[:, None, :] == targets[:, :, None]              # [b,t,i]: y_i == y_t
            lp_copy = torch.logsumexp(la.masked_fill(~(match & allowed), NEG), -1)   # (B, T)
            lp = torch.logaddexp(F.logsigmoid(-g) + lp_vocab, F.logsigmoid(g) + lp_copy)
        return -lp.mean()

    def forward(self, idx, targets=None, caches=None, pos: int = 0):
        if caches is None:
            h = self._hidden(idx)
            if targets is None:
                return self.lm_head(h)                                      # vocab-only logits
            return self.copy_loss(h, targets)
        # ---- generation: returns LOG-PROBS of the mixture for the last position, (B, 1, V) ----
        h = self._hidden(idx, caches, pos)
        cc, T = caches[-1], idx.size(1)
        cc.k[:, pos:pos + T] = self.copy_k(h).to(cc.k.dtype)
        cc.ids[:, pos:pos + T] = idx
        t = pos + T - 1                                                     # current position
        if pos == 0:
            self._prompt_len = T                                            # remember where the prompt ends
        hl = h[:, -1:]
        z = self.lm_head(hl).float()
        with torch.autocast("cuda", enabled=False):
            out = torch.log_softmax(z, -1)                                  # t=0: nothing to copy
            if t >= 1:
                hf = hl.float()
                g = self.gate(hf) + getattr(self, "copy_gate_bias", 0.0)   # decode-time gate shift
                out = out + F.logsigmoid(-g)
                # copy_prompt_only: point only at prompt tokens (no copying of the model's own output)
                n = min(t, self._prompt_len - 1) if getattr(self, "copy_prompt_only", False) else t
                if n >= 1:
                    a = torch.softmax(self.copy_q(hf) @ cc.k[:, :n].float().transpose(1, 2)
                                      / math.sqrt(self.cfg.copy_dim), -1)   # (B, 1, n), keys i < n
                    pc = torch.zeros_like(out).scatter_add_(2, cc.ids[:, None, 1:n + 1], a)
                    out = torch.logaddexp(out, F.logsigmoid(g) + torch.log(pc))
            out = out + torch.logsumexp(z, -1, keepdim=True)            # logit scale, as M1
        return out

In [ ]:
%%writefile src/model_registry.py
"""build_model(arch, cfg): the four Phase-1 architectures (same registry as the pre-training script)."""


def build_model(arch: str, model_cfg: dict):
    if arch == "dense":                                   # M1
        from model_dense import DenseGPT, DenseConfig
        return DenseGPT(DenseConfig(**model_cfg))
    if arch == "mla_moe":                                 # M2
        from model_mla_moe import MLAMoEGPT, MLAMoEConfig
        return MLAMoEGPT(MLAMoEConfig(**model_cfg))
    if arch == "encdec":                                  # M3
        from model_encdec import EncDecLM, EncDecConfig
        return EncDecLM(EncDecConfig(**model_cfg))
    if arch == "copy":                                    # M4
        from model_copy import CopyGPT, CopyConfig
        return CopyGPT(CopyConfig(**model_cfg))
    raise ValueError(f"unknown arch {arch}")

In [ ]:
%%writefile src/p2_ner.py
"""
p2_ner.py -- Phase 2 (Legal NER) data pipeline: exact character offsets <-> model tokens.

Why this file exists
  Our Phase-1 models were trained on NORMALISED text (ASCII only, single spaces, no newlines),
  but L-NER is scored on character offsets of the ORIGINAL text, strictly (one wrong character
  = wrong entity). So we normalise each document while keeping, for every normalised character,
  the index of the original character it came from. Tokens are then mapped back exactly.

Pipeline
  normalise_with_map(text)          -> norm, idx   (idx[j] = original index of norm[j])
  encode_doc(text, tk)              -> token ids + each token's ORIGINAL char span (core, no
                                       leading space)
  bio_tags(spans, entities)         -> one BIO tag id per token (O, B-X, I-X; 25 tags)
  tags_to_entities(tags, spans)     -> [start, end, label] in original offsets
  make_windows(n, win, stride)      -> overlapping windows over long documents
  strict_f1(pred, gold)             -> micro P/R/F1 + per-label F1 (exact span + label)
"""
from __future__ import annotations

import hashlib
import json
import unicodedata
from collections import Counter, defaultdict

from common import PUNCT_MAP

LABELS = ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT',
          'STAT', 'PREC', 'DATE', 'CASENO']
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]     # 25 BIO tags
TAG2ID = {t: i for i, t in enumerate(TAGS)}


# ----------------------------------------------------------------------------- normalisation
def _norm_char(c: str) -> str:
    """Same rule as common.normalise, applied to ONE character (whitespace handled by caller)."""
    if c.isascii():
        return c
    s = unicodedata.normalize("NFKD", c.translate(PUNCT_MAP))
    return s.encode("ascii", "ignore").decode("ascii")


def normalise_with_map(text: str):
    """Return (norm, idx): norm == common.normalise(text) and idx[j] is the original index of
    norm[j]. A whitespace run becomes one space mapped to the run's first character."""
    out, idx, pending_space = [], [], None
    for i, ch in enumerate(text):
        if ch.isspace():
            if out and pending_space is None:
                pending_space = i                          # remember where the run started
            continue
        rep = _norm_char(ch)
        if not rep:
            continue                                      # dropped char (e.g. zero-width)
        if rep.isspace() or not rep.strip():
            if out and pending_space is None:
                pending_space = i
            continue
        if pending_space is not None:
            out.append(" "); idx.append(pending_space); pending_space = None
        for r in rep:
            if r.isspace():
                continue
            out.append(r); idx.append(i)
    return "".join(out), idx


# ----------------------------------------------------------------------------- tokens <-> chars
def encode_doc(text: str, tk, split_punct: bool = True):
    """Tokenise the normalised document. Returns (ids, spans) where spans[k] = (start, end) of
    token k's CORE (leading space stripped) in ORIGINAL offsets, or None for a pure-space token.
    split_punct: our BPE glues punctuation runs into one token (e.g. '.[' or '].'), which can
    straddle an entity boundary ('... & Ors.' + '[1]'). Such tokens are re-encoded one character
    at a time so every punctuation mark gets its own token and boundary."""
    norm, idx = normalise_with_map(text)
    enc = tk.encode(norm, add_special_tokens=False)
    ids, spans = [], []
    for tid, (s, e) in zip(enc.ids, enc.offsets):
        c = s
        while c < e and norm[c] == " ":
            c += 1                                        # " court" -> core "court"
        core = norm[c:e]
        if split_punct and len(core) > 1 and not any(ch.isalnum() for ch in core):
            for j, ch in enumerate(core):                 # one token per punctuation mark
                piece = (" " if j == 0 and c > s else "") + ch
                for sub in tk.encode(piece, add_special_tokens=False).ids:
                    ids.append(sub)
                    is_space = tk.decode([sub]).strip() == ""
                    spans.append(None if is_space else (idx[c + j], idx[c + j] + 1))
            continue
        ids.append(tid)
        spans.append((idx[c], idx[e - 1] + 1) if c < e else None)
    return ids, spans


def bio_tags(spans, entities):
    """BIO tag id per token. A token is B-X if its core starts exactly at an entity start, I-X if
    it lies inside that entity, else O."""
    begin, inside = {}, {}
    for s, e, lab, *_ in entities:
        begin[s] = lab
        for c in range(s, e):
            inside[c] = (lab, s)
    tags = []
    for sp in spans:
        if sp is None:
            tags.append(0); continue
        s, _ = sp
        if s in begin:
            tags.append(TAG2ID[f"B-{begin[s]}"])
        elif s in inside:
            tags.append(TAG2ID[f"I-{inside[s][0]}"])
        else:
            tags.append(0)
    return tags


def tags_to_entities(tags, spans):
    """Decode BIO tags back to [start, end, label] (original offsets). An I-X that does not
    continue an X entity starts a new one (standard repair)."""
    ents, cur = [], None
    for t, sp in zip(tags, spans):
        if sp is None:
            continue                                      # spaces never break or extend a span
        tag = TAGS[t]
        if tag == "O":
            if cur: ents.append(cur); cur = None
            continue
        p, lab = tag.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: ents.append(cur)
            cur = [sp[0], sp[1], lab]
        else:
            cur[1] = sp[1]                                # extend the current entity
    if cur:
        ents.append(cur)
    return ents


def make_windows(n_tokens: int, win: int = 512, stride: int = 256):
    """Start offsets of overlapping windows covering [0, n_tokens)."""
    if n_tokens <= win:
        return [0]
    starts = list(range(0, n_tokens - win, stride))
    return starts + [n_tokens - win]


# ----------------------------------------------------------------------------- data + metric
def load_fold(text_path, gold_path=None, fold=""):
    T = json.load(open(text_path))
    G = json.load(open(gold_path)) if gold_path else {}
    return [{"id": str(k), "text": v, "ents": G.get(k, []), "fold": fold} for k, v in T.items()]


def dev_split(docs, n_dev_per_fold=5):
    """Deterministic held-out set: the n docs per fold with the smallest id-hash."""
    h = lambda d: hashlib.md5(d["id"].encode()).hexdigest()
    dev = set()
    for f in sorted({d["fold"] for d in docs}):
        dev |= {d["id"] for d in sorted((d for d in docs if d["fold"] == f), key=h)[:n_dev_per_fold]}
    return [d for d in docs if d["id"] not in dev], [d for d in docs if d["id"] in dev]


def strict_f1(pred: dict, gold: dict):
    """pred/gold: {doc_id: [[s, e, label, ...], ...]}. Exact span AND label must match."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for d in set(gold) | set(pred):
        G = {(e[0], e[1], e[2]) for e in gold.get(d, [])}
        P = {(e[0], e[1], e[2]) for e in pred.get(d, [])}
        for x in P & G: tp[x[2]] += 1
        for x in P - G: fp[x[2]] += 1
        for x in G - P: fn[x[2]] += 1
    def f(a, b, c):
        p = a / max(1, a + b); r = a / max(1, a + c)
        return p, r, 2 * p * r / max(1e-9, p + r)
    P_, R_, F_ = f(sum(tp.values()), sum(fp.values()), sum(fn.values()))
    per = {l: round(f(tp[l], fp[l], fn[l])[2], 4) for l in LABELS}
    return {"precision": P_, "recall": R_, "f1": F_, "per_label_f1": per}


# ----------------------------------------------------------------------------- v3: constrained Viterbi decoding
def _allowed():
    """allowed[i, j]: tag j may follow tag i (BIO rule: I-X only after B-X or I-X)."""
    import numpy as np
    K = len(TAGS); A = np.ones((K, K), dtype=bool)
    for j, t in enumerate(TAGS):
        if t.startswith("I-"):
            lab = t[2:]
            A[:, j] = False
            A[TAG2ID[f"B-{lab}"], j] = A[TAG2ID[f"I-{lab}"], j] = True
    return A


def viterbi_bio(logp, spans):
    """Most probable VALID BIO sequence given per-token log-probs (T, K). Pure-space tokens
    (span None) are skipped, so they can neither break nor start an entity."""
    import numpy as np
    keep = [i for i, sp in enumerate(spans) if sp is not None]
    out = [0] * len(spans)
    if not keep:
        return out
    lp = np.asarray(logp)[keep]                                   # (n, K)
    A = _allowed(); NEG = -1e9
    trans = np.where(A, 0.0, NEG)                                 # (K, K)
    start = np.array([NEG if t.startswith("I-") else 0.0 for t in TAGS])
    n, K = lp.shape
    score = start + lp[0]; back = np.zeros((n, K), dtype=np.int32)
    for t in range(1, n):
        cand = score[:, None] + trans                             # prev x cur
        back[t] = cand.argmax(0)
        score = cand.max(0) + lp[t]
    best = [int(score.argmax())]
    for t in range(n - 1, 0, -1):
        best.append(int(back[t, best[-1]]))
    for i, tag in zip(keep, reversed(best)):
        out[i] = tag
    return out


# ----------------------------------------------------------------------------- v4: word-consistent Viterbi
def word_groups(spans, text):
    """Group sub-word tokens into words: token k continues the previous token's word if they touch
    (no space) and both sides are letters/digits (e.g. 'VENKAT' + 'ACH' + 'ALA')."""
    groups, prev = [], None
    for k, sp in enumerate(spans):
        if sp is None:
            continue
        if (prev is not None and sp[0] == prev[1] and text[prev[1] - 1].isalnum()
                and text[sp[0]].isalnum()):
            groups[-1].append(k)
        else:
            groups.append([k])
        prev = sp
    return groups


def _cont():
    """Tag that a word's continuation pieces must take: B-X/I-X -> I-X, O -> O."""
    return [0 if t == "O" else TAG2ID["I-" + t[2:]] for t in TAGS]


def viterbi_words(logp, spans, text):
    """Constrained Viterbi over WORDS: every sub-word piece of a word gets a consistent tag
    (first piece B-X or I-X, the rest I-X; or all O). A word's score for tag t is the exact joint
    log-prob of that consistent assignment, so a name can no longer be cut inside a word."""
    import numpy as np
    lp = np.asarray(logp)
    groups = word_groups(spans, text)
    out = [0] * len(spans)
    if not groups:
        return out
    cont = np.array(_cont())
    W = np.stack([lp[g[0]] + (lp[g[1:]][:, cont].sum(0) if len(g) > 1 else 0.0) for g in groups])
    A = _allowed(); NEG = -1e9
    trans = np.where(A, 0.0, NEG)
    start = np.array([NEG if t.startswith("I-") else 0.0 for t in TAGS])
    n = len(groups)
    score = start + W[0]; back = np.zeros((n, len(TAGS)), dtype=np.int32)
    for t in range(1, n):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0); score = cand.max(0) + W[t]
    best = [int(score.argmax())]
    for t in range(n - 1, 0, -1):
        best.append(int(back[t, best[-1]]))
    for g, tag in zip(groups, reversed(best)):
        out[g[0]] = tag
        for k in g[1:]:
            out[k] = int(cont[tag])
    return out

In [ ]:
%%writefile src/p2_model.py
"""
p2_model.py -- Phase 2 token tagger on top of ANY Phase-1 backbone (M1 dense, M2 MLA+MoE,
M3 encoder-decoder, M4 copy head) + windowed training / prediction.

  build_tagger(ckpt, bidirectional)  -> TokenTagger: backbone hidden states -> Linear -> 25 BIO tags
      * M1/M2/M4 decoders: tok_emb -> blocks -> norm_f (the LM head is not used)
      * M3: its bidirectional ENCODER only
      * bidirectional=True switches the decoders' attention from causal to full (every token sees
        its right context too, which matters for entity boundaries; cf. LLM2Vec, 2024)
  make_examples(docs, tk)            -> (ids, tags) windows for training
  predict_docs(model, docs, tk)      -> {doc_id: [[start, end, label], ...]} in original offsets
  train_tagger(...)                  -> fine-tunes, keeps the best dev-F1 weights
"""
from __future__ import annotations

import contextlib
import math
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import p2_ner as ner

_SDPA = F.scaled_dot_product_attention


@contextlib.contextmanager
def full_attention():
    """Inside this block every scaled_dot_product_attention call ignores is_causal, so causal
    decoders attend in both directions. All our attention modules call F.scaled_dot_product_attention
    at run time, so swapping the function on the module is enough."""
    def sdpa(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=False, **kw):
        return _SDPA(q, k, v, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=False, **kw)
    F.scaled_dot_product_attention = sdpa
    try:
        yield
    finally:
        F.scaled_dot_product_attention = _SDPA


class TokenTagger(nn.Module):
    def __init__(self, backbone, arch: str, n_tags: int = len(ner.TAGS), dropout: float = 0.1,
                 bidirectional: bool = False):
        super().__init__()
        self.backbone, self.arch, self.bidirectional = backbone, arch, bidirectional
        d = backbone.cfg.d_model
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d, n_tags)
        nn.init.normal_(self.head.weight, 0.0, 0.02); nn.init.zeros_(self.head.bias)

    def hidden(self, ids):
        bb = self.backbone
        if self.arch == "encdec":
            return bb.encode(ids)                                   # M3: bidirectional encoder
        x = bb.tok_emb(ids)                                         # M1 / M2 / M4 decoders
        for blk in bb.blocks:
            x = blk(x, bb.rope_cos, bb.rope_sin)
        return bb.norm_f(x)

    def forward(self, ids):
        ctx = full_attention() if (self.bidirectional and self.arch != "encdec") else contextlib.nullcontext()
        with ctx:
            h = self.hidden(ids)
        return self.head(self.drop(h))                               # (B, T, n_tags)


def build_tagger(ckpt_path: str, bidirectional: bool = False, dropout: float = 0.1):
    """Load a Phase-1 best.pt and wrap it as a tagger (backbone weights in fp32)."""
    from model_registry import build_model
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    bb = build_model(ck["arch"], ck["model_cfg"])
    bb.load_state_dict({k: v.float() for k, v in ck["model"].items()})
    if ck["arch"] == "encdec":
        del bb.dec                                                   # decoder is not needed for tagging
    return TokenTagger(bb, ck["arch"], dropout=dropout, bidirectional=bidirectional)


# ----------------------------------------------------------------------------- data
def encode_all(docs, tk):
    """Tokenise every document once: {id: (ids, spans, tags)}."""
    out = {}
    for d in docs:
        ids, spans = ner.encode_doc(d["text"], tk)
        out[d["id"]] = (ids, spans, ner.bio_tags(spans, d["ents"]))
    return out


def make_examples(enc, doc_ids, win=512, stride=256):
    ex = []
    for i in doc_ids:
        ids, _, tags = enc[i]
        for s in ner.make_windows(len(ids), win, stride):
            ex.append((ids[s:s + win], tags[s:s + win]))
    return ex


def _batch(examples, device):
    L = max(len(x) for x, _ in examples)
    ids = torch.zeros(len(examples), L, dtype=torch.long)
    tags = torch.full((len(examples), L), -100, dtype=torch.long)     # -100 = padding, ignored
    for b, (x, t) in enumerate(examples):
        ids[b, :len(x)] = torch.tensor(x); tags[b, :len(t)] = torch.tensor(t)
    return ids.to(device), tags.to(device)


# ----------------------------------------------------------------------------- prediction
@torch.no_grad()
def predict_logits(model, ids, win=512, stride=256, device="cuda", bs=16):
    """Per-token log-probs for one document. Each token takes its prediction from the window in
    which it has the most context: most central for bidirectional models, latest position (most
    left context) for causal decoders."""
    model.eval()
    causal = model.arch != "encdec" and not model.bidirectional
    starts = ner.make_windows(len(ids), win, stride)
    out = torch.zeros(len(ids), model.head.out_features)
    best = torch.full((len(ids),), -1e9)
    for b in range(0, len(starts), bs):
        chunk = starts[b:b + bs]
        x, _ = _batch([(ids[s:s + win], ids[s:s + win]) for s in chunk], device)
        with torch.autocast("cuda", dtype=torch.float16):
            lp = model(x).float().log_softmax(-1).cpu()
        for r, s in enumerate(chunk):
            n = min(win, len(ids) - s)
            j = torch.arange(n, dtype=torch.float)
            score = j if causal else torch.minimum(j, n - 1 - j)   # how much context position j has
            take = score > best[s:s + n]
            out[s:s + n][take] = lp[r, :n][take]
            best[s:s + n] = torch.maximum(best[s:s + n], score)
    return out


def predict_docs(model, docs, enc, win=512, stride=256, device="cuda"):
    preds = {}
    for d in docs:
        ids, spans, _ = enc[d["id"]]
        tags = predict_logits(model, ids, win, stride, device).argmax(-1).tolist()
        preds[d["id"]] = ner.tags_to_entities(tags, spans)
    return preds


# ----------------------------------------------------------------------------- training
def train_tagger(model, train_ex, dev_docs, enc, epochs=5, lr=5e-5, head_lr=1e-3, bs=8,
                 weight_decay=0.01, warmup=0.06, class_weight=None, device="cuda", seed=0, log=print):
    """AdamW, linear warm-up + linear decay, fp16 autocast. Evaluates strict F1 on dev_docs after
    every epoch and restores the best epoch's trainable weights at the end."""
    torch.manual_seed(seed); random.seed(seed)
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    head_ids = {id(p) for p in model.head.parameters()}
    groups = [{"params": [p for p in params if id(p) not in head_ids], "lr": lr},
              {"params": [p for p in params if id(p) in head_ids], "lr": head_lr}]
    opt = torch.optim.AdamW(groups, weight_decay=weight_decay)
    steps = epochs * math.ceil(len(train_ex) / bs); warm = max(1, int(warmup * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda")
    w = None if class_weight is None else class_weight.to(device)
    best_f1, best_state, hist, step = -1.0, None, [], 0
    n_train = sum(p.numel() for p in params)
    log(f"trainable params: {n_train/1e6:.2f}M | {len(train_ex)} windows | {steps} steps")
    for ep in range(epochs):
        model.train(); random.shuffle(train_ex); t0 = time.time(); tot = 0.0
        for b in range(0, len(train_ex), bs):
            ids, tags = _batch(train_ex[b:b + bs], device)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(ids)
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), tags.view(-1),
                                   weight=w, ignore_index=-100)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step(); step += 1   # (first-step lr warning is harmless)
            tot += loss.item()
        r = ner.strict_f1(predict_docs(model, dev_docs, enc, device=device),
                          {d["id"]: d["ents"] for d in dev_docs})
        hist.append({"epoch": ep + 1, "loss": tot / math.ceil(len(train_ex) / bs), **r})
        log(f"epoch {ep+1}: loss {hist[-1]['loss']:.4f} | dev strict F1 {r['f1']:.4f} "
            f"(P {r['precision']:.3f} R {r['recall']:.3f}) | {time.time()-t0:.0f}s")
        if r["f1"] > best_f1:
            best_f1 = r["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return hist, best_f1


# ----------------------------------------------------------------------------- v2: Viterbi decoding + ignore space tokens
def encode_all(docs, tk):
    """Tokenise every document once: {id: (ids, spans, tags)}. Pure-space tokens get tag -100
    (ignored in the loss): they carry no characters, so they have no entity label."""
    out = {}
    for d in docs:
        ids, spans = ner.encode_doc(d["text"], tk)
        tags = [t if sp is not None else -100 for t, sp in zip(ner.bio_tags(spans, d["ents"]), spans)]
        out[d["id"]] = (ids, spans, tags)
    return out


DECODE = "viterbi"                      # "argmax" (independent per token) or "viterbi" (valid BIO only)


def predict_docs(model, docs, enc, win=512, stride=256, device="cuda", decode=None):
    decode = decode or DECODE
    preds = {}
    for d in docs:
        ids, spans, _ = enc[d["id"]]
        lp = predict_logits(model, ids, win, stride, device)
        tags = ner.viterbi_bio(lp.numpy(), spans) if decode == "viterbi" else lp.argmax(-1).tolist()
        preds[d["id"]] = ner.tags_to_entities(tags, spans)
    return preds


# ----------------------------------------------------------------------------- v3: document header as context
HEADER = 128                            # first HEADER tokens of the document are prepended to every window
SEP_ID = 3                              # <|sep|> (reserved special token from our A1-BPE tokenizer)


def _with_header(ids, s, win, header):
    """Window [s, s+win) with the document header in front (only when the window does not
    already start at the beginning). Returns (input ids, offset of the window inside them)."""
    if header and s > 0:
        return ids[:header] + [SEP_ID] + ids[s:s + win], header + 1
    return ids[s:s + win], 0


def make_examples(enc, doc_ids, win=512, stride=256, header=None):
    header = HEADER if header is None else header
    ex = []
    for i in doc_ids:
        ids, _, tags = enc[i]
        for s in ner.make_windows(len(ids), win, stride):
            x, off = _with_header(ids, s, win, header)
            ex.append((x, [-100] * off + tags[s:s + win]))        # header tokens are not trained on
    return ex


@torch.no_grad()
def predict_logits(model, ids, win=512, stride=256, device="cuda", bs=16, header=None):
    header = HEADER if header is None else header
    model.eval()
    causal = model.arch != "encdec" and not model.bidirectional
    starts = ner.make_windows(len(ids), win, stride)
    out = torch.zeros(len(ids), model.head.out_features)
    best = torch.full((len(ids),), -1e9)
    for b in range(0, len(starts), bs):
        chunk = starts[b:b + bs]
        packed = [_with_header(ids, s, win, header) for s in chunk]
        x, _ = _batch([(p, p) for p, _ in packed], device)
        with torch.autocast("cuda", dtype=torch.float16):
            lp = model(x).float().log_softmax(-1).cpu()
        for r, (s, (_, off)) in enumerate(zip(chunk, packed)):
            n = min(win, len(ids) - s)
            j = torch.arange(n, dtype=torch.float)
            score = j if causal else torch.minimum(j, n - 1 - j)
            take = score > best[s:s + n]
            out[s:s + n][take] = lp[r, off:off + n][take]
            best[s:s + n] = torch.maximum(best[s:s + n], score)
    return out


# ----------------------------------------------------------------------------- v4: word-level decoding
DECODE = "words"                        # "argmax" | "viterbi" (token level) | "words" (word-consistent)


def predict_docs(model, docs, enc, win=512, stride=256, device="cuda", decode=None):
    decode = decode or DECODE
    preds = {}
    for d in docs:
        ids, spans, _ = enc[d["id"]]
        lp = predict_logits(model, ids, win, stride, device).numpy()
        if decode == "words":
            tags = ner.viterbi_words(lp, spans, d["text"])
        elif decode == "viterbi":
            tags = ner.viterbi_bio(lp, spans)
        else:
            tags = lp.argmax(-1).tolist()
        preds[d["id"]] = ner.tags_to_entities(tags, spans)
    return preds

In [ ]:
%%writefile src/p2_peft.py
"""
p2_peft.py -- adaptation methods for the Phase-2 tagger (all four backbones) + knowledge distillation.

  apply_method(tagger, method, **kw) freezes the backbone and adds the method's trainable parts
    "full"      : everything trainable (reference)
    "partial"   : top-k transformer layers + head
    "normgain"  : BitFit-style. Our models have no bias vectors, so the analogous tiny parameter
                  set is the RMSNorm gains (~19k params) + head
    "lora"      : W x + (a/r) B A x                     on query + value projections (Hu et al. 2022)
    "lorma_plus": (I + (a/r) B A) W x  = y + (a/r) B (A y),  B=0, A random   (LoRMA+, Bihany et al. 2025)
    "lorma_pi"  : I_pi(B A) W x, rows of BA cyclically rotated (row i by i); B[:,0]=1, A[0,0]=1, so
                  I_pi(BA) = I at start (LoRMA-pi)
    "adapter"   : bottleneck adapter after every layer, y + U relu(D y), U=0 (Houlsby/Pfeiffer)
    "prompt"    : P trainable soft-prompt vectors prepended to the input (Lester et al. 2021)
  train_kd(student, teacher, ...)   : CE + T^2 KL(teacher || student) on soft tag distributions
  shrink_dense(tagger, keep)        : student = M1/M4 with only the given layers kept (DistilBERT init)
"""
from __future__ import annotations

import math
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import p2_ner as ner
import p2_model as pm

TARGETS = ("wq", "wv", "w_uv")          # query + value projections (w_uv = MLA's value up-projection)


def _layers(tagger):
    bb = tagger.backbone
    return bb.enc if tagger.arch == "encdec" else bb.blocks


def count(tagger):
    tot = sum(p.numel() for p in {id(p): p for p in tagger.parameters()}.values())
    tr = sum(p.numel() for p in tagger.parameters() if p.requires_grad)
    return tr, tot


# ----------------------------------------------------------------------------- low-rank layers
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=8):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(x, self.A), self.B)


class LoRMAPlusLinear(nn.Module):
    """h = (I + (a/r) BA) W0 x  -- computed as y + s * B (A y) with y = W0 x (cost like LoRA)."""
    def __init__(self, base: nn.Linear, r=8, alpha=8):
        super().__init__()
        d = base.out_features
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))                       # B = 0 -> identity at start

    def forward(self, x):
        y = self.base(x)
        return y + self.s * F.linear(F.linear(y, self.A), self.B)


class LoRMAPiLinear(nn.Module):
    """h = I_pi(BA) W0 x, I_pi rotates row i of BA cyclically by i (rank inflation)."""
    def __init__(self, base: nn.Linear, r=8):
        super().__init__()
        d = base.out_features
        self.base = base
        B = torch.randn(d, r) * 0.02; B[:, 0] = 1.0                      # first column of B = ones
        A = torch.zeros(r, d); A[0, 0] = 1.0                             # A[0,0] = 1, rest 0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A)
        i = torch.arange(d)
        self.register_buffer("rot", (i[None, :] - i[:, None]) % d, persistent=False)  # new[i,j]=old[i,j-i]

    def forward(self, x):
        M = (self.B @ self.A).gather(1, self.rot)                        # I_pi(BA), = I at init
        return F.linear(self.base(x), M.to(x.dtype))                    # (M y) for every token


def _wrap_linears(tagger, make):
    n = 0
    for layer in _layers(tagger):
        for mod in list(layer.modules()):              # snapshot: we replace children
            for name in TARGETS:
                lin = getattr(mod, name, None)
                if isinstance(lin, nn.Linear):
                    setattr(mod, name, make(lin)); n += 1
    return n


# ----------------------------------------------------------------------------- adapters + prompts
class AdapterLayer(nn.Module):
    """Wraps a transformer layer: out = layer(x, ...) ; out + U relu(D out) (U = 0 at start)."""
    def __init__(self, layer, d, bottleneck=64):
        super().__init__()
        self.layer = layer
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, 0.0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x, *args, **kw):
        y = self.layer(x, *args, **kw)
        return y + self.up(F.relu(self.down(y)))


class PromptEmbedding(nn.Module):
    def __init__(self, emb: nn.Embedding, n_prompt=20):
        super().__init__()
        self.emb, self.n = emb, n_prompt
        idx = torch.randint(1000, 30000, (n_prompt,))                   # init from real token embeddings
        self.prompt = nn.Parameter(emb.weight[idx].detach().clone())

    def forward(self, ids):
        x = self.emb(ids)
        return torch.cat([self.prompt.unsqueeze(0).expand(x.size(0), -1, -1).to(x.dtype), x], 1)


class DropPrefix(nn.Module):
    """Head wrapper: drops the soft-prompt positions before tagging."""
    def __init__(self, head, n):
        super().__init__()
        self.head, self.n, self.out_features = head, n, head.out_features

    def forward(self, h):
        return self.head(h[:, self.n:])


# ----------------------------------------------------------------------------- dispatcher
def apply_method(tagger, method, r=8, alpha=8, top_k=4, bottleneck=64, n_prompt=20):
    if method == "full":
        return tagger
    for p in tagger.backbone.parameters():
        p.requires_grad = False
    if method == "partial":
        for layer in list(_layers(tagger))[-top_k:]:
            for p in layer.parameters():
                p.requires_grad = True
    elif method == "normgain":
        for n_, p in tagger.backbone.named_parameters():
            if "norm" in n_:
                p.requires_grad = True
    elif method == "lora":
        _wrap_linears(tagger, lambda l: LoRALinear(l, r, alpha))
    elif method == "lorma_plus":
        _wrap_linears(tagger, lambda l: LoRMAPlusLinear(l, r, alpha))
    elif method == "lorma_pi":
        _wrap_linears(tagger, lambda l: LoRMAPiLinear(l, r))
    elif method == "adapter":
        L = _layers(tagger); d = tagger.backbone.cfg.d_model
        for i in range(len(L)):
            L[i] = AdapterLayer(L[i], d, bottleneck)
    elif method == "prompt":
        bb = tagger.backbone
        bb.tok_emb = PromptEmbedding(bb.tok_emb, n_prompt)
        tagger.head = DropPrefix(tagger.head, n_prompt)
    else:
        raise ValueError(method)
    for p in tagger.head.parameters():
        p.requires_grad = True
    return tagger


# ----------------------------------------------------------------------------- distillation
def shrink_dense(tagger, keep=(0, 2, 4, 6, 8, 10)):
    """Student from a decoder tagger: keep only the listed layers (every other layer, as in
    DistilBERT's initialisation)."""
    bb = tagger.backbone
    bb.blocks = nn.ModuleList([bb.blocks[i] for i in keep])
    return tagger


def train_kd(student, teacher, train_ex, dev_docs, enc, epochs=5, lr=1e-4, head_lr=1e-3, bs=8,
             T=2.0, lam=0.5, device="cuda", seed=0, log=print):
    """loss = (1-lam) * CE(student, gold) + lam * T^2 * KL(teacher_T || student_T), padding ignored."""
    torch.manual_seed(seed); random.seed(seed)
    student.to(device); teacher.to(device).eval()
    params = [p for p in student.parameters() if p.requires_grad]
    head_ids = {id(p) for p in student.head.parameters()}
    opt = torch.optim.AdamW([{"params": [p for p in params if id(p) not in head_ids], "lr": lr},
                             {"params": [p for p in params if id(p) in head_ids], "lr": head_lr}],
                            weight_decay=0.01)
    steps = epochs * math.ceil(len(train_ex) / bs); warm = max(1, int(0.06 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda")
    best_f1, best_state, hist = -1.0, None, []
    log(f"student trainable {sum(p.numel() for p in params)/1e6:.2f}M | {len(train_ex)} windows")
    for ep in range(epochs):
        student.train(); random.shuffle(train_ex); t0 = time.time(); tot = 0.0
        for b in range(0, len(train_ex), bs):
            ids, tags = pm._batch(train_ex[b:b + bs], device)
            mask = tags != -100
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                t_logits = teacher(ids).float()
            with torch.autocast("cuda", dtype=torch.float16):
                s_logits = student(ids).float()
            ce = F.cross_entropy(s_logits[mask], tags[mask])
            kl = F.kl_div(F.log_softmax(s_logits[mask] / T, -1), F.log_softmax(t_logits[mask] / T, -1),
                          log_target=True, reduction="batchmean") * T * T
            loss = (1 - lam) * ce + lam * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
        r = ner.strict_f1(pm.predict_docs(student, dev_docs, enc, device=device),
                          {d["id"]: d["ents"] for d in dev_docs})
        hist.append({"epoch": ep + 1, "loss": tot / math.ceil(len(train_ex) / bs), **r})
        log(f"epoch {ep+1}: loss {hist[-1]['loss']:.4f} | dev strict F1 {r['f1']:.4f} "
            f"(P {r['precision']:.3f} R {r['recall']:.3f}) | {time.time()-t0:.0f}s")
        if r["f1"] > best_f1:
            best_f1 = r["f1"]; best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
    student.load_state_dict(best_state)
    return hist, best_f1

## 2 · Setup: data, tokenisation, class weights, macro-F1 selection

* **Data:** the L-NER folds (`fold*-text.json` = {doc_id: text}, `fold*-gold.json` = {doc_id: [[start, end, label], …]}).
  5 documents per training fold are held out as dev (deterministic md5 split); the remaining documents train.
* **Windows:** 512 tokens, stride 256; each token's prediction comes from the window where it has the most context.
* **Class weights:** inverse √frequency of the 25 BIO tags (capped at 10) – rare labels matter for macro-F1.
* **Model selection:** the epoch with the best dev **strict macro-F1** (mean over labels present in the dev gold).

In [ ]:
# ============================================================================================
# Setup — paths (EDIT THESE), data, tokenisation, class weights, metric helpers
# ============================================================================================
import os, sys, glob, json, zipfile
import numpy as np, torch

SRC       = os.path.abspath("src")                                  # modules written above
DATA      = "/path/to/Phase_2_LNER"                                 # folder with fold*-text.json / fold*-gold.json (+ Test/)
TOKENIZER = "/path/to/hf_a1_bpe_v2_32k.json"                        # tokenizers-format JSON of the A1 BPE tokenizer
CKPT = {                                                            # Phase-1 pre-trained checkpoints (best.pt)
    "m1": "/path/to/checkpoints/m1_dense_100m/best.pt",
    "m2": "/path/to/checkpoints/m2_mla_moe/best.pt",
    "m3": "/path/to/checkpoints/m3_encdec/best.pt",
    "m4": "/path/to/checkpoints/m4_copy/best.pt",
    "abl_base":   "/path/to/checkpoints/abl_base/best.pt",          # 100M-token ablation backbones
    "abl_noqk":   "/path/to/checkpoints/abl_noqk/best.pt",
    "abl_untied": "/path/to/checkpoints/abl_untied/best.pt",
}
OUT = "submissions"; os.makedirs(OUT, exist_ok=True)
DEV = "cuda:0" if torch.cuda.is_available() else "cpu"

sys.path.insert(0, SRC)
import p2_ner as ner, p2_model as pm, p2_peft as pp
from tokenizers import Tokenizer
pm.HEADER = 0                                                       # no header context (tested: no gain)

# ---- documents: training folds -> train/dev split; test fold has no gold ---------------------
tk = Tokenizer.from_file(TOKENIZER)
docs = []
for tp in sorted(glob.glob(f"{DATA}/**/*text*.json", recursive=True)):
    if "/Test/" in tp:
        continue
    gp = next(tp.replace("text", x) for x in ("gold", "labels", "label") if os.path.exists(tp.replace("text", x)))
    docs += ner.load_fold(tp, gp, fold=os.path.basename(tp).split("-")[0])
train_docs, dev_docs = ner.dev_split(docs, 5)
test_path = glob.glob(f"{DATA}/Test/*text*.json")[0]
test_docs = [{"id": str(k), "text": v, "ents": []} for k, v in json.load(open(test_path)).items()]
print(f"{len(train_docs)} train / {len(dev_docs)} dev / {len(test_docs)} test documents")

# ---- tokenise once; training windows -----------------------------------------------------------
ENC = pm.encode_all(train_docs + dev_docs + test_docs, tk)          # {doc_id: (ids, char spans, BIO tags)}
TRAIN_EX = pm.make_examples(ENC, [d["id"] for d in train_docs], header=0)
print(f"{len(TRAIN_EX)} training windows of <=512 tokens")

# ---- class weights: inverse sqrt frequency of each BIO tag (capped) -----------------------------
cnt = np.bincount([t for _, ts in TRAIN_EX for t in ts if t >= 0], minlength=len(ner.TAGS)).astype(float)
CW = torch.tensor(np.where(cnt > 0, np.sqrt(cnt.max() / np.maximum(cnt, 1)), 1.0), dtype=torch.float).clamp(max=10.0)

# ---- strict macro-F1 over the labels present in the gold; used for epoch selection ------------
_strict = ner.strict_f1
def macro_of(r, gold):
    present = {e[2] for es in gold.values() for e in es}
    return float(np.mean([r["per_label_f1"][l] for l in ner.LABELS if l in present]))

def strict_f1_macro(pred, gold):                                   # train_tagger / train_kd select on r["f1"]
    r = _strict(pred, gold); r["micro"] = r["f1"]; r["f1"] = macro_of(r, gold); return r

def evaluate(model):
    gold = {d["id"]: d["ents"] for d in dev_docs}
    r = _strict(pm.predict_docs(model, dev_docs, ENC, device=DEV), gold)
    return {"macro": round(macro_of(r, gold), 4), "micro": round(r["f1"], 4)}

## 3 · Experiment driver

`run(...)` builds a tagger on a Phase-1 backbone, optionally drops layers (student), applies the adaptation
method, trains (plain or with a teacher for KD), reports dev strict macro/micro-F1 and writes a competition zip
(`prediction.json` = {doc_id: [[start, end, label], …]} at the zip root).

In [ ]:
# ============================================================================================
# Experiment driver
# ============================================================================================
def write_zip(model, name):
    """Predict the test documents and write submissions/p2_<name>.zip with prediction.json at the root."""
    pred = pm.predict_docs(model, test_docs, ENC, device=DEV)
    folder = f"{OUT}/p2_{name}"; os.makedirs(folder, exist_ok=True)
    json.dump({d: [[s, e, l] for s, e, l in ents] for d, ents in pred.items()}, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p2_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    print(f"   wrote {folder}/p2_{name}.zip ({sum(map(len, pred.values()))} entities)")


def run(name, backbone, method, lr, epochs=5, bidirectional=False, keep_layers=None,
        teacher=None, class_weight=True, seed=0):
    """backbone: key of CKPT | method: full, normgain, lora, lorma_plus, lorma_pi, adapter, prompt
    keep_layers: e.g. (0, 2, 4, 6, 8, 10) -> 6-layer student | teacher: trained tagger -> KD."""
    model = pm.build_tagger(CKPT[backbone], bidirectional=bidirectional)
    if keep_layers:
        pp.shrink_dense(model, keep_layers)                         # DistilBERT-style layer dropping
    pp.apply_method(model, method)
    tr, tot = pp.count(model)
    print(f"\n=== {name}: {backbone} | {method} | trainable {tr/1e6:.3f}M of {tot/1e6:.1f}M ===")
    ner.strict_f1 = strict_f1_macro                                 # select the epoch by dev MACRO-F1
    try:
        if teacher is None:
            pm.train_tagger(model, list(TRAIN_EX), dev_docs, ENC, epochs=epochs, lr=lr, head_lr=1e-3, bs=8,
                            class_weight=CW if class_weight else None, device=DEV, seed=seed)
        else:
            pp.train_kd(model, teacher, list(TRAIN_EX), dev_docs, ENC, epochs=epochs, lr=lr,
                        T=2.0, lam=0.5, device=DEV, seed=seed)
    finally:
        ner.strict_f1 = _strict
    r = evaluate(model)
    print(f"   dev strict macro-F1 {r['macro']:.4f} | micro {r['micro']:.4f}")
    write_zip(model, name)
    return model

## 4 · The 11 runs

Learning rates per method: 5e-4 for low-rank / adapter methods, 1e-3 for norm gains, 1e-2 for soft prompts,
5e-5 for full fine-tuning of 100M-parameter backbones (1e-4 for the 6-layer students). 5 epochs each, best epoch
by dev macro-F1. Each run takes roughly 5–10 minutes on a single T4.

**KD pair:** the teacher is M1 + LoRA (100.4M parameters). The student keeps 6 of M1's 12 layers (62.4M,
−38% parameters, ~20% faster inference) and is trained with 0.5·CE + 0.5·T²·KL(teacher‖student), T = 2.
The control is the identical student trained without the teacher.

In [ ]:
# ============================================================================================
# Parameter-efficient methods on the Phase-1 models
# ============================================================================================
run("m1_normgain_cw", "m1", "normgain",   lr=1e-3)      # only RMSNorm gains + head (BitFit analogue)
run("m1_prompt_cw",   "m1", "prompt",     lr=1e-2)      # 20 trainable soft-prompt vectors + head
run("m1_lorma_pi",    "m1", "lorma_pi",   lr=5e-4)      # h = I_pi(BA) W0 x  (rank-inflated, identity at init)
run("m2_lorma_plus",  "m2", "lorma_plus", lr=5e-4)      # h = (I + s*BA) W0 x on MLA query/value projections
run("m4_adapter",     "m4", "adapter",    lr=5e-4)      # bottleneck adapters (64) after every layer
run("m3_full",        "m3", "full",       lr=5e-5)      # encoder of the encoder-decoder, fully fine-tuned

In [ ]:
# ============================================================================================
# Knowledge distillation: M1+LoRA teacher -> 6-layer M1 student, and the no-teacher control
# ============================================================================================
teacher = run("m1_lora_teacher", "m1", "lora", lr=5e-4, class_weight=False)             # 100.4M teacher
STUDENT = dict(backbone="m1", method="full", lr=1e-4, keep_layers=(0, 2, 4, 6, 8, 10))
run("kd_student_m1x6",   teacher=teacher, **STUDENT)                                      # CE + T^2 KL
run("nokd_student_m1x6", **STUDENT)                                                       # same student, CE only
del teacher; torch.cuda.empty_cache()

In [ ]:
# ============================================================================================
# Backbone ablations (100M-token pre-training each): same LoRA recipe on three M1 variants
#   pre-training SC val loss: base 3.427 | no QK-norm 3.270 | untied embeddings 3.293
# ============================================================================================
for b in ["abl_base", "abl_noqk", "abl_untied"]:
    run(f"{b}_lora_cw", b, "lora", lr=5e-4)

## Notes and observations

* **Capacity ordering:** across methods the score roughly follows the number of trainable parameters
  (full ≈ LoRA family > adapters > norm gains ≈ prompt), but on the hidden test set the tiny methods (norm gains,
  prompt tuning) were much closer to LoRA than the 10-document dev set suggested – small dev sets are noisy.
* **LoRMA:** multiplicative low-rank updates matched LoRA on other tasks but were weaker here (LoRMAπ in particular).
* **KD:** the 6-layer student with or without the teacher scores about the same on test (0.506 vs 0.511) – the
  dropped layers cost more than the soft labels give back on 60 training documents.
* **M3:** the 6-layer encoder of the encoder-decoder is the weakest backbone for token tagging.
* **Ablations:** removing QK-norm improved both pre-training loss and downstream F1 at a 100M-token budget
  (QK-norm bounds the initial attention logits and slows early learning).